# 코드잇 스프린트 미션16: 이미지 모델 밴치마크 사이트 개발(모델링)
---
여러분들은 지금까지 AI 스프린트 미션들을 수행하며 다양한 모델들을 학습 및 구현해보셨습니다. 
이번 미션에서는 그 모델들을 다시 가져와서, 여러 형태의 포맷으로 모델을 변환하여 저장해보는 실습을 해봅시다.




## 가이드라인
1. 이전 미션에서 다루었던 모델들 중 하나 이상을 자유롭게 선택하여 모델 학습을 진행합니다.
2. 아래의 3가지 타입의 모델로 변환하여 저장해봅시다.
    - `.pth` (PyTorch 기본 저장 형식)
    - `.pth` (양자화 된 버전)
    - `.onnx` (ONNX 형식)

## 데이터셋

`mnist data set`

- **데이터 구성**:
    - **학습**: 60,000장
    - **테스트**: 10,000장
    - **크기**: 28×28 grayscale
    - **클래스**: 0-9 숫자

**용량**: ~12MB (매우 작음)

## 사용 모델

`ViT`

# 학습 코드

## 1. 라이브러리

In [12]:
%pwd

'c:\\codeit_mission\\mission_16'

In [16]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: c:\Users\hambu\.pyenv\pyenv-win\versions\3.12.3\python.exe -m pip install --upgrade pip


In [37]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time

In [38]:
class Config:
    """모델 및 학습 설정"""
    # 데이터
    data_root = './data'
    batch_size = 64
    num_workers = 2
    
    # 모델 아키텍처
    img_size = 28
    patch_size = 4
    in_channels = 1
    num_classes = 10
    embed_dim = 128
    depth = 6
    num_heads = 8
    mlp_ratio = 4.0
    dropout = 0.1
    
    # 학습
    num_epochs = 10
    learning_rate = 0.001
    momentum = 0.9
    
    # 디바이스
    device = 'cuda' if torch.cuda.is_available() else 'cpu'


## 2. 데이터 로드

In [39]:
def load_data(config):
    """MNIST 데이터셋 로드 및 DataLoader 생성"""
    print("=" * 60)
    print("데이터 로드 중...")
    print("=" * 60)
    
    train_dataset = datasets.MNIST(
        root=config.data_root,
        train=True,
        download=True,
        transform=transforms.ToTensor()
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        pin_memory=torch.cuda.is_available()
    )
    
    print(f"✅ 학습 데이터: {len(train_dataset)}개")
    print(f"✅ 배치 수: {len(train_loader)}")
    print(f"✅ 배치 크기: {config.batch_size}")
    
    return train_loader

## 3. ViT 모델 코드 작성

In [40]:
class PatchEmbedding(nn.Module):
    """이미지 → 패치 임베딩"""
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.projection = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )
        
    def forward(self, x):
        x = self.projection(x)
        x = x.flatten(2).transpose(1, 2)
        return x

In [41]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention"""
    def __init__(self, embed_dim, num_heads, dropout):
        super().__init__()
        assert embed_dim % num_heads == 0
        
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        B, N, C = x.shape
        
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)
        
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.dropout(x)
        
        return x

In [42]:
class MLP(nn.Module):
    """Feed-Forward Network"""
    def __init__(self, embed_dim, mlp_ratio, dropout):
        super().__init__()
        hidden_dim = int(embed_dim * mlp_ratio)
        
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.act = nn.GELU()
        self.dropout1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        self.dropout2 = nn.Dropout(dropout)
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.dropout2(x)
        return x

In [43]:
class TransformerBlock(nn.Module):
    """Transformer Encoder Block"""
    def __init__(self, embed_dim, num_heads, mlp_ratio, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim, mlp_ratio, dropout)
        
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [44]:
class VisionTransformer(nn.Module):
    """Vision Transformer"""
    def __init__(self, config):
        super().__init__()
        
        self.patch_embed = PatchEmbedding(
            config.img_size, config.patch_size,
            config.in_channels, config.embed_dim
        )
        num_patches = self.patch_embed.num_patches
        
        self.cls_token = nn.Parameter(torch.zeros(1, 1, config.embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, config.embed_dim))
        self.pos_drop = nn.Dropout(config.dropout)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(
                config.embed_dim, config.num_heads,
                config.mlp_ratio, config.dropout
            )
            for _ in range(config.depth)
        ])
        
        self.norm = nn.LayerNorm(config.embed_dim)
        self.head = nn.Linear(config.embed_dim, config.num_classes)
        
        self._init_weights()
        
    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LayerNorm):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        B = x.shape[0]
        
        x = self.patch_embed(x)
        
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        x = x + self.pos_embed
        x = self.pos_drop(x)
        
        for block in self.blocks:
            x = block(x)
        
        x = self.norm(x)
        cls_token_final = x[:, 0]
        logits = self.head(cls_token_final)
        
        return logits


## 4. 모델 학습

In [45]:
def train_model(model, train_loader, config):
    """모델 학습"""
    print("\n" + "=" * 60)
    print("학습 시작")
    print("=" * 60)
    
    model = model.to(config.device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=config.learning_rate,
        momentum=config.momentum
    )
    
    for epoch in range(config.num_epochs):
        model.train()
        
        running_loss = 0.0
        correct = 0
        total = 0
        
        start_time = time.time()
        
        print(f"\n[Epoch {epoch+1}/{config.num_epochs}]")
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(config.device), labels.to(config.device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            if (batch_idx + 1) % 200 == 0:
                avg_loss = running_loss / (batch_idx + 1)
                acc = 100. * correct / total
                print(f"  Batch [{batch_idx+1}/{len(train_loader)}] "
                      f"Loss: {avg_loss:.4f} | Acc: {acc:.2f}%")
        
        epoch_time = time.time() - start_time
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100. * correct / total
        
        print(f"✅ Epoch {epoch+1} 완료 - "
              f"Loss: {epoch_loss:.4f} | "
              f"Acc: {epoch_acc:.2f}% | "
              f"Time: {epoch_time:.1f}s")
    
    print("\n" + "=" * 60)
    print("학습 완료!")
    print("=" * 60)
    
    return model

In [46]:
def get_model_info(model):
    """모델 정보 출력"""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print("\n" + "=" * 60)
    print("모델 정보")
    print("=" * 60)
    print(f"총 파라미터: {total_params:,}")
    print(f"학습 가능 파라미터: {trainable_params:,}")
    print(f"디바이스: {next(model.parameters()).device}")


## 메인 실행

In [47]:
def main():
    """전체 파이프라인 실행"""
    # 설정
    config = Config()
    
    print("=" * 60)
    print("Vision Transformer (ViT) for MNIST")
    print("=" * 60)
    print(f"디바이스: {config.device}")
    print(f"배치 크기: {config.batch_size}")
    print(f"학습 Epoch: {config.num_epochs}")
    print(f"학습률: {config.learning_rate}")
    model_name = input("저장할 모델 이름을 입력하세요.:")
    
    # 1. 데이터 로드
    train_loader = load_data(config)
    
    # 2. 모델 생성
    print("\n" + "=" * 60)
    print("모델 생성 중...")
    print("=" * 60)
    model = VisionTransformer(config)
    get_model_info(model)
    
    # 3. 학습
    trained_model = train_model(model, train_loader, config)
    
    # 4. 모델 저장
    print("\n" + "=" * 60)
    print("모델 저장 중...")
    print("=" * 60)
    
    torch.save(trained_model.state_dict(), f'./models/mission_16_{model_name}.pth')
    print(f"✅ 모델 저장 완료: mission_16_{model_name}.pth")


In [48]:
if __name__ == "__main__":
    main()

Vision Transformer (ViT) for MNIST
디바이스: cpu
배치 크기: 64
학습 Epoch: 10
학습률: 0.001
데이터 로드 중...
✅ 학습 데이터: 60000개
✅ 배치 수: 938
✅ 배치 크기: 64

모델 생성 중...

모델 정보
총 파라미터: 1,199,882
학습 가능 파라미터: 1,199,882
디바이스: cpu

학습 시작

[Epoch 1/10]
  Batch [200/938] Loss: 2.1564 | Acc: 19.44%
  Batch [400/938] Loss: 2.0886 | Acc: 21.96%
  Batch [600/938] Loss: 2.0491 | Acc: 23.32%
  Batch [800/938] Loss: 2.0259 | Acc: 24.04%
✅ Epoch 1 완료 - Loss: 2.0120 | Acc: 24.54% | Time: 454.9s

[Epoch 2/10]
  Batch [200/938] Loss: 1.9268 | Acc: 27.16%
  Batch [400/938] Loss: 1.9001 | Acc: 27.98%
  Batch [600/938] Loss: 1.8682 | Acc: 29.38%
  Batch [800/938] Loss: 1.8403 | Acc: 30.48%
✅ Epoch 2 완료 - Loss: 1.8221 | Acc: 31.09% | Time: 541.2s

[Epoch 3/10]
  Batch [200/938] Loss: 1.5493 | Acc: 41.19%
  Batch [400/938] Loss: 1.4779 | Acc: 44.42%
  Batch [600/938] Loss: 1.4198 | Acc: 47.22%
  Batch [800/938] Loss: 1.3580 | Acc: 49.76%
✅ Epoch 3 완료 - Loss: 1.3176 | Acc: 51.46% | Time: 530.7s

[Epoch 4/10]
  Batch [200/938] Loss: 1